# Z3-Rust 18 — Sudoku 4x4 : comparaison des modes `Array` et `Constants`

**Navigation** : [Index Z3-Python](README.md) | [Index SMT](../README.md) | [Index SymbolicAI](../../README.md) | [Série Z3 C#](../Z3-Linq2Z3/README.md) | [<< 17 - Array Theory](Z3-17-Array-Theory-Python.ipynb) | [→ 19 - Optimisation multi-objectif](Z3-19-Optimization-Lexicographic-Python.ipynb) | [Jumeau C#](Z3-18-Sudoku-Modes-CSharp.ipynb) | [Version Python](Z3-18-Sudoku-Modes-Python.ipynb)

> **Solveur** : Z3 — crate Rust `z3` 0.23.0, exécuté dans le noyau `rust` (evcxr). Le portage est **intégral** : même modèle, mêmes contraintes, mêmes verdicts que le jumeau Python.

## Objectifs d'apprentissage

À la fin de ce notebook, vous saurez :
1. Modéliser le **même** Sudoku 4x4 de **deux façons** différentes en Z3 — un encodage par `Int` constants (un par cellule), un encodage par `Array` (théorie des tableaux).
2. Énoncer les **contraintes canoniques** du Sudoku 4x4 (domaine `1..4`, rangées/colonnes/blocs 2x2 à valeurs distinctes) dans chaque mode.
3. **Vérifier que les deux encodages convergent** vers une solution valide (carré latin), et comprendre pourquoi le choix d'encodage n'affecte pas la sémantique mais peut affecter la performance.

Ce notebook est le port Rust du carnet Python [`Z3-18-Sudoku-Modes-Python.ipynb`](Z3-18-Sudoku-Modes-Python.ipynb). Il le complète par le carnet [Z3-Python-02 Sudoku](Z3-02-Sudoku-Python.ipynb) (qui résout le Sudoku dans un seul mode) en montrant que **deux modélisations équivalentes** du même problème mènent au même résultat — une leçon clé sur la séparation entre **problème** (les contraintes) et **encodage** (comment on les exprime).

> **Différence de moteur, à connaître** : le jumeau Python utilise le paquet PyPI `z3-solver` (pyz3 4.16.0), le jumeau C# le paquet NuGet `Microsoft.Z3` (4.12.2), et ce jumeau la crate Rust `z3` 0.23.0, qui lie la bibliothèque native **Z3 5.3.3** du poste. Les trois côtés branchent le **même cœur Z3** par trois bindings officiels ; le cœur est donc différent d'une version à l'autre, la sémantique identique — c'est ce que mesure le registre de conformité différentielle (`scripts/notebook_tools/rust_conformance.d/z3-18-sudoku-modes.yaml`).

In [2]:
:dep z3 = "0.23.0"

// Le noyau `rust` (evcxr) execute chaque cellule dans le meme etat persistant :
// les `fn` et les `const` d'une cellule restent disponibles dans les suivantes,
// les variables locales (`let`) et les `use` non. D'ou le patron du carnet :
// chaque cellule de calcul definit ses propres fonctions (qui persistent) et
// reconstruit son etat local a partir des items persistants.

use z3::ast::{Ast, Int};
use z3::{SatResult, Solver};

// Taille de la grille et taille de bloc (racine carree de N).
const N: usize = 4;
const B: usize = 2;

println!("Imports OK : z3 crate 0.23.0 (libz3 {})", z3::full_version());

Imports OK : z3 crate 0.23.0 (libz3 5.3.3.0 430d19e697e2ec9bad79ff210d41b4a7a5fcfa3e 430d19e)


### Lecture du résultat

La sortie confirme que l'environnement est prêt : `Imports OK : z3 crate 0.23.0 (libz3 Z3 5.3.3)` — la crate `z3` est bien liée, les cellules suivantes peuvent poser leurs contraintes sur Z3.

Une remarque de version : la bibliothèque native liée par `z3-sys` (5.3.3) n'est pas la même que le paquet Python `z3-solver` (4.16.0) du jumeau Python. **Le cœur Z3 diffère d'une version à l'autre** ; la sémantique SMT, elle, est stable — c'est ce que le registre de conformité différentielle vérifie, ligne à ligne.

## 1. Vue d'ensemble — un même Sudoku, deux modes d'encodage

Un Sudoku 4x4 se modélise par **16 cellules** prenant des valeurs dans `{1, 2, 3, 4}`, soumises à trois familles de contraintes : chaque **rangée**, chaque **colonne** et chaque **bloc 2x2** contient des valeurs **distinctes**. Z3 permet d'exprimer ces 16 cellules de deux manières fondamentalement différentes :

| Mode | Encodage pyz3 | Encodage crate `z3` | Analogie |
|------|----------------|---------------------|----------|
| **Constants** | 16 variables `Int` séparées (`c_0_0`, `c_0_1`, …, `c_3_3`) | 16 constantes `Int` nommées (`Int::new_const("c_0_0")`) | un tableau de scalaires |
| **Array** | un `Array(Int, Int)` indexé par `r*4+c` (théorie des tableaux) | un `Array(Int, Int)` indexé par `r*4+c` (`Array::new_const` + `select`) | une fonction Index → Valeur |

Les deux modes sont **sémantiquement équivalents** : ils expriment le même espace de solutions. La différence est **structurelle** — le mode `Array` traite la grille comme une fonction (utile si on raisonne sur des cellules arbitraires ou des mises à jour `Store`), le mode `Constants` traite la grille comme un tableau de scalaires (plus direct, plus lisible). Ce carnet résout le **même** Sudoku dans les deux modes et vérifie qu'ils convergent.

## 2. La grille 4x4 et ses contraintes canoniques

Fixons d'abord une instance avec quelques **givens** (cellules pré-remplies) pour garantir une solution unique et pouvoir comparer. La grille suivante a quatre givens ; le solveur doit compléter le reste.

```
+---+---+---+---+
| 1 | . | . | . |
+---+---+---+---+
| . | . | . | 3 |
+---+---+---+---+
| . | . | 2 | . |
+---+---+---+---+
| . | 4 | . | . |
+---+---+---+---+
```

Les contraintes canoniques (identiques dans les deux modes) :
1. **Domaine** : chaque cellule ∈ `{1, 2, 3, 4}`.
2. **Rangées** : les 4 cellules de chaque rangée sont distinctes.
3. **Colonnes** : les 4 cellules de chaque colonne sont distinctes.
4. **Blocs 2x2** : les 4 cellules de chaque bloc sont distinctes.
5. **Givens** : les cellules pré-remplies sont fixées.

## 3. Résolution en mode `Constants` (un `Int` par cellule)

Le mode le plus direct : on crée 16 variables `Int` organisées en matrice 4x4, puis on pose les contraintes Sudoku canoniques. C'est l'encodage idiomatique de la crate `z3` comme de pyz3.

Quatre différences d'API avec pyz3, à connaître avant de lire le code :

| Python (pyz3) | Rust (crate `z3` 0.23) | Raison |
|---------------|------------------------|--------|
| `Int(f"c_{r}_{c}")` | `Int::new_const(format!("c_{r}_{c}"))` | le nom est un `Symbol` dans la crate |
| `CONST[r][c] == v` | `Ast::eq(&CONST[r][c], &Int::from_i64(v))` | `==` en Rust est une comparaison **structurelle**, pas une contrainte SMT ; `Ast::eq` est la contrainte |
| `CONST[r][c] >= 1` | `CONST[r][c].ge(&Int::from_i64(1))` | les comparaisons sont des méthodes **inhérentes** de `Int`, pas des méthodes du trait `Ast` |
| `Distinct([...])` | `Int::distinct(&ligne[..])` | `distinct` est une méthode **statique** du trait : le type d'impl doit être nommé, sinon le compilateur ne peut pas inférer `Self` |

La contrainte d'égalité est le point qui surprend le plus : en Python, `==` construit une formule SMT ; en Rust, `==` compare deux valeurs. Une contrainte SMT s'écrit donc `Ast::eq(&a, &b)` — c'est l'équivalent exact du `==` de pyz3. Deux pièges de compilation méritent d'être signalés, parce qu'ils ne sont pas devinables : `Ast::eq` doit être appelé par son nom complet (la méthode `eq` existe aussi dans `PartialEq`, que la crate implémente — l'appel court `a.eq(b)` serait ambigu) ; et `Ast::distinct` sans le type d'impl est refusé par le compilateur (erreur E0790), il faut écrire `Int::distinct`.

In [3]:
use z3::ast::{Ast, Int};
use z3::{SatResult, Solver};

// Mode Constants : 16 Int constants (matrice 4x4).
// Les `fn` persistent d'une cellule a l'autre sous evcxr : la cellule 9
// (mode Array) et la cellule 12 (verification) reutilisent celles-ci.

fn givens() -> Vec<((usize, usize), i64)> {
    vec![((0, 0), 1), ((1, 3), 3), ((2, 2), 2), ((3, 1), 4)]
}

// Les 16 variables Int nommees, comme les 16 scalaires de pyz3.
fn constantes() -> Vec<Vec<Int>> {
    (0..N)
        .map(|r| {
            (0..N)
                .map(|c| Int::new_const(format!("c_{}_{}", r, c)))
                .collect()
        })
        .collect()
}

// Une rangee, une colonne ou un bloc : les valeurs sont Distinct.
// `distinct` est une methode STATIQUE du trait `Ast` : le type d'impl doit
// etre nomme (`Int::distinct`), faute de quoi le compilateur ne peut pas
// inferer `Self` (erreur E0790).
fn toutes_distinctes(s: &Solver, valeurs: &[&Int]) {
    s.assert(&Int::distinct(valeurs));
}

// La grille solution du mode Constants : (statut, grille).
fn grille_constantes() -> (String, Vec<Vec<i64>>) {
    let cellules = constantes();
    let s = Solver::new();

    // (1) Domaine 1..4 pour chaque cellule.
    for r in 0..N {
        for c in 0..N {
            s.assert(&cellules[r][c].ge(&Int::from_i64(1)));
            s.assert(&cellules[r][c].le(&Int::from_i64(N as i64)));
        }
    }
    // (2) Rangees : valeurs distinctes.
    for r in 0..N {
        let ligne: Vec<&Int> = (0..N).map(|c| &cellules[r][c]).collect();
        toutes_distinctes(&s, &ligne);
    }
    // (3) Colonnes : valeurs distinctes.
    for c in 0..N {
        let colonne: Vec<&Int> = (0..N).map(|r| &cellules[r][c]).collect();
        toutes_distinctes(&s, &colonne);
    }
    // (4) Blocs 2x2 : valeurs distinctes (N=4 -> blocs de taille 2).
    for br in (0..N).step_by(B) {
        for bc in (0..N).step_by(B) {
            let mut bloc: Vec<&Int> = Vec::new();
            for i in 0..B {
                for j in 0..B {
                    bloc.push(&cellules[br + i][bc + j]);
                }
            }
            toutes_distinctes(&s, &bloc);
        }
    }
    // (5) Givens : (rangee, colonne) -> valeur.
    for ((r, c), v) in givens() {
        s.assert(&Ast::eq(&cellules[r][c], &Int::from_i64(v)));
    }

    let statut = match s.check() {
        SatResult::Sat => "sat",
        SatResult::Unsat => "unsat",
        SatResult::Unknown => "unknown",
    };
    let m = s.get_model().unwrap();
    let mut grille = Vec::new();
    for r in 0..N {
        let mut ligne = Vec::new();
        for c in 0..N {
            ligne.push(m.eval(&cellules[r][c], true).unwrap().as_i64().unwrap());
        }
        grille.push(ligne);
    }
    (statut.to_string(), grille)
}

// L'affichage : la meme forme que le jumeau Python ("   [1, 3, 4, 2]").
fn afficher(grille: &Vec<Vec<i64>>) {
    for ligne in grille {
        let rendu: Vec<String> = ligne.iter().map(|v| v.to_string()).collect();
        println!("   [{}]", rendu.join(", "));
    }
}

let (statut, grille_const) = grille_constantes();
println!("Mode Constants - statut : {}", statut);
println!("Solution (Constants) :");
afficher(&grille_const);

Mode Constants - statut : sat


Solution (Constants) :


   [1, 3, 4, 2]


   [4, 2, 1, 3]


   [3, 1, 2, 4]


   [2, 4, 3, 1]


### Lecture du résultat

Le mode `Constants` répond `Mode Constants - statut : sat` et imprime la grille solution, dont la première rangée `[1, 3, 4, 2]` respecte le given `c_0_0 == 1` posé plus haut.

La correspondance avec pyz3 est directe : `Int::new_const("c_0_0")` est la variable nommée, `Solver::new()` est le solveur, `s.assert(...)` est un `add(...)`, et `Ast::distinct(&[...])` est le `Distinct(...)` de la liste Python. Le modèle se lit par `m.eval(&variable, true)` — le `true` active la complétion du modèle, comme le `model_completion` de pyz3.

## 4. Résolution en mode `Array` (théorie des tableaux)

Le mode alternatif : la grille entière est **un seul** `Array(Int, Int)` où l'indice `r*4 + c` désigne la cellule `(r, c)`. On accède aux cellules via `Select` (voir [Z3-Python-17 Array Theory](Z3-17-Array-Theory-Python.ipynb)). Les contraintes sont les mêmes, mais exprimées sur des `Select(grid, i)` plutôt que sur des scalaires.

Une différence d'API mérite d'être notée : en pyz3, `Select(grid, i)` renvoie directement un `Int` ; en Rust, la crate renvoie un `Dynamic` (le type dynamique de Z3, utilisable pour toute sorte). `as_int()` le convertit — c'est le seul déréférencement spécifique au type dans tout le carnet.

In [4]:
use z3::ast::{Array, Ast, Int};
use z3::{SatResult, Solver};

// Mode Array : un Array(Int, Int) indexe par r*4+c.
// Le `select` de la crate renvoie un `Dynamic` ; `as_int()` le ramene au
// type `Int` attendu par les contraintes arithmetiques.

fn cellule(grille: &Array, r: usize, c: usize) -> Int {
    let index = Int::from_i64((r * N + c) as i64);
    grille.select(&index).as_int().unwrap()
}

// La grille solution du mode Array : (statut, grille).
fn grille_array() -> (String, Vec<Vec<i64>>) {
    let tri_int = Int::from_i64(0).get_sort();
    let grille = Array::new_const("grid", &tri_int, &tri_int);
    let s = Solver::new();

    // (1) Domaine 1..4 pour chaque cellule.
    for r in 0..N {
        for c in 0..N {
            let v = cellule(&grille, r, c);
            s.assert(&v.ge(&Int::from_i64(1)));
            s.assert(&v.le(&Int::from_i64(N as i64)));
        }
    }
    // (2) Rangees distinctes.
    // Les valeurs sont d'abord collectees dans un `Vec<Int>` POSSESSIONNE :
    // une reference `&cellule(...)`designee un temporaire qui serait libere
    // a la fin de l'instruction (erreur de borrow checker).
    for r in 0..N {
        let valeurs: Vec<Int> = (0..N).map(|c| cellule(&grille, r, c)).collect();
        let ligne: Vec<&Int> = valeurs.iter().collect();
        s.assert(&Int::distinct(&ligne[..]));
    }
    // (3) Colonnes distinctes.
    for c in 0..N {
        let valeurs: Vec<Int> = (0..N).map(|r| cellule(&grille, r, c)).collect();
        let colonne: Vec<&Int> = valeurs.iter().collect();
        s.assert(&Int::distinct(&colonne[..]));
    }
    // (4) Blocs 2x2 distincts.
    for br in (0..N).step_by(B) {
        for bc in (0..N).step_by(B) {
            let mut valeurs: Vec<Int> = Vec::new();
            for i in 0..B {
                for j in 0..B {
                    valeurs.push(cellule(&grille, br + i, bc + j));
                }
            }
            let bloc: Vec<&Int> = valeurs.iter().collect();
            s.assert(&Int::distinct(&bloc[..]));
        }
    }
    // (5) Givens (memes que le mode Constants).
    for ((r, c), v) in givens() {
        let cell = cellule(&grille, r, c);
        s.assert(&Ast::eq(&cell, &Int::from_i64(v)));
    }

    let statut = match s.check() {
        SatResult::Sat => "sat",
        SatResult::Unsat => "unsat",
        SatResult::Unknown => "unknown",
    };
    let m = s.get_model().unwrap();
    // La grille solution est un `Vec<Vec<i64>>` — il ne faut PAS le nommer
    // `grille` comme le tableau symbolique, sinon il le masque et le
    // dereferencement `&grille` vise le Vec au lieu du Array.
    let mut solution = Vec::new();
    for r in 0..N {
        let mut ligne = Vec::new();
        for c in 0..N {
            let cell = cellule(&grille, r, c);
            ligne.push(m.eval(&cell, true).unwrap().as_i64().unwrap());
        }
        solution.push(ligne);
    }
    (statut.to_string(), solution)
}

let (statut, grille_arr) = grille_array();
println!("Mode Array - statut : {}", statut);
println!("Solution (Array) :");
afficher(&grille_arr);

Mode Array - statut : sat


Solution (Array) :


   [1, 3, 4, 2]


   [4, 2, 1, 3]


   [3, 1, 2, 4]


   [2, 4, 3, 1]


### Lecture du résultat

Le mode `Array` répond à son tour `Mode Array - statut : sat` avec la même première rangée `[1, 3, 4, 2]` : les deux encodages convergent déjà ici, avant la vérification formelle de la cellule suivante.

## 5. Vérification — les deux solutions sont des carrés latins valides (et coïncident)

Les deux modes doivent produire des grilles qui satisfont **toutes** les contraintes Sudoku (domaine, rangées, colonnes, blocs, givens). Vérifions-le programmMATiquement, puis comparons les deux solutions. Avec les mêmes givens et (probablement) une solution unique, elles doivent coïncider exactement.

Détail d'implémentation : sous evcxr, les variables locales d'une cellule ne survivent pas à la cellule. La vérification **recalcule donc les deux grilles** en appelant les fonctions de la cellule 6 et de la cellule 9 — celles-ci, étant des `fn`, persistent.

In [5]:
// Verification : chaque solution est un carre latin Sudoku 4x4 valide.
// Les variables locales ne persistent pas sous evcxr : les deux grilles sont
// recalculees a partir des fonctions (persistantes) des cellules 6 et 9.

// Rendu du booleen : Rust ecrit `true`/`false`, Python ecrit `True`/`False`
// et C# ecrit `True`. Le rendu est aligne sur le cote de reference (le jumeau
// Python) pour que la comparaison differentielle porte sur la SEMANTIQUE et
// non sur la convention de casse d'un langage.
fn booleen(b: bool) -> &'static str {
    if b {
        "True"
    } else {
        "False"
    }
}

fn est_valide_sudoku(sol: &Vec<Vec<i64>>) -> bool {
    // Domaine + valeurs entieres 1..4.
    if !sol
        .iter()
        .flatten()
        .all(|v| (1..=N as i64).contains(v))
    {
        return false;
    }
    // Rangees.
    for ligne in sol.iter() {
        let mut vus: Vec<i64> = ligne.clone();
        vus.sort_unstable();
        vus.dedup();
        if vus.len() != N {
            return false;
        }
    }
    // Colonnes.
    for c in 0..N {
        let mut vus: Vec<i64> = (0..N).map(|r| sol[r][c]).collect();
        vus.sort_unstable();
        vus.dedup();
        if vus.len() != N {
            return false;
        }
    }
    // Blocs 2x2.
    for br in (0..N).step_by(B) {
        for bc in (0..N).step_by(B) {
            let mut vus: Vec<i64> = Vec::new();
            for i in 0..B {
                for j in 0..B {
                    vus.push(sol[br + i][bc + j]);
                }
            }
            vus.sort_unstable();
            vus.dedup();
            if vus.len() != N {
                return false;
            }
        }
    }
    // Givens respectes.
    for ((r, c), v) in givens() {
        if sol[r][c] != v {
            return false;
        }
    }
    true
}

let (_, grille_const) = grille_constantes();
let (_, grille_arr) = grille_array();

println!(
    "Solution Constants valide : {}",
    booleen(est_valide_sudoku(&grille_const))
);
println!(
    "Solution Array valide      : {}",
    booleen(est_valide_sudoku(&grille_arr))
);
println!(
    "Les deux solutions coïncident : {}",
    booleen(grille_const == grille_arr)
);

Solution Constants valide : True


Solution Array valide      : True


Les deux solutions coïncident : True


### Interprétation — pourquoi les deux modes convergent

Les deux solutions coïncident sur les **16 indices de la grille**, avec la correspondance bijective `(r, c) <-> r*N + c` entre le mode `Constants` et le mode `Array` : c'est le témoin que, sur cette instance, **l'encodage ne change pas la solution calculée**. Deux précautions : (i) deux solutions identiques ne prouvent pas l'égalité des *ensembles* de solutions — seul le dénombrement (une solution unique par mode, puis comparaison) le ferait ; (ii) en mode `Array`, les entrées du tableau hors des 16 indices de la grille restent non contraintes — elles ne participent ni au modèle ni à la comparaison. Le mode `Constants` et le mode `Array` sont deux **syntaxes** différentes pour le même ensemble de contraintes — le solveur Z3 les réécrit toutes deux vers le même noyau SMT avant la résolution.

Le choix entre les deux modes est donc une décision de **lisibilité et de contexte**, pas de correction :
- **`Constants`** est plus lisible pour une grille fixe (on nomme chaque cellule), et c'est l'encodage idiomatique de la crate `z3` comme de pyz3.
- **`Array`** est pertinent dès qu'on veut raisonner sur des **cellules arbitraires** (indice calculé) ou des **mises à jour** (`Store` pour simuler un coup), ou raisonner sur une grille **non bornée**.

**Verdict honnête (Prong-B)** : sur un Sudoku 4x4, les deux modes se résolvent instantanément — aucune différence de performance observable à l'œil nu ici, et ce carnet n'en mesure aucune (l'exercice 3 laisse l'instrument au lecteur). Si une différence doit exister, ce sera sur des grilles plus grandes (9x9) ou quand l'encodage `Array` permet des raccourcis de modélisation qu'une explosion de `Constants` (81 variables nommées) rendrait illisibles — hypothèse à trancher empiriquement par les 100 runs de l'exercice 3.

## Exercices

Les exercices vous font réutiliser les deux modes sur de nouvelles instances. Chaque exercice suit le squelette : choisir un mode, poser les contraintes canoniques (domaine, rangées, colonnes, blocs, givens), appeler `check()` puis lire le modèle.

**Rappel C.1** : les stubs ne lèvent **jamais** d'erreur — ils se contentent d'afficher l'énoncé. Le notebook s'exécute de bout en bout même exercices non résolus.

### Exercice 1 — Ajouter un hint dans les deux modes

Reprenez la grille 4x4 ci-dessus et ajoutez un **cinquième given** (par exemple `c_0_2 == 4`) dans **les deux modes** (Constants et Array). Vérifiez que les deux solutions coïncident toujours. La solution change-t-elle par rapport à la grille à 4 givens ?

**Indice** : il suffit d'ajouter une contrainte `Ast::eq(...)` supplémentaire dans chaque solveur, puis de relancer `check()`.

In [6]:
// EXERCICE 1 : ajouter un 5e given dans les deux modes.
// TODO etudiant :
// 1. Reprendre grille_constantes() et grille_array(), ajouter c_0_2 == 4 et cell(0,2) == 4
// 2. s.check() / s_arr.check(), lire les modeles
// 3. Comparer : les deux solutions coincident-elles ? differentes de la grille a 4 givens ?
println!("Exercice 1 a completer : ajouter un 5e given (c_0_2 == 4) dans les deux modes et comparer.");

Exercice 1 a completer : ajouter un 5e given (c_0_2 == 4) dans les deux modes et comparer.


### Exercice 2 — Sudoku 9x9 en mode `Constants`

Généralisez le mode `Constants` à un Sudoku **9x9** (valeurs `1..9`, blocs 3x3). Posez les contraintes canoniques pour une grille **vide** (sans givens) et demandez une solution au solveur. Affichez la grille complète obtenue.

**Indice** : `N = 9`, `B = 3`, même structure de boucles (domaine, rangées `Distinct`, colonnes, blocs). Une grille vide a de nombreuses solutions — le solveur en produit une.

In [7]:
// EXERCICE 2 : Sudoku 9x9 en mode Constants (grille vide).
// TODO etudiant :
// 1. N = 9, B = 3 ; construire 81 variables Int (x_r_c)
// 2. Domaine 1..9, rangees/colonnes/blocs 3x3 Distinct
// 3. s.check(), afficher la grille 9x9 solution
println!("Exercice 2 a completer : Sudoku 9x9 mode Constants, grille vide -> une solution.");

Exercice 2 a completer : Sudoku 9x9 mode Constants, grille vide -> une solution.


### Exercice 3 — Comparaison empirique des deux modes

Mesurez le **temps de résolution** (`Instant::now()` avant/après `check()`) du Sudoku 4x4 dans chacun des deux modes. Lancez chaque résolution disons 100 fois et comparez les temps moyens. Le mode `Array` est-il plus lent, plus rapide, ou équivalent au mode `Constants` ?

**Question subsidiaire** : pourquoi le mode `Array` pourrait-il être (légèrement) plus lent sur une grille fixe, alors qu'il est plus expressif sur une grille dynamique ?

In [8]:
// EXERCICE 3 : comparaison empirique des temps de resolution (Constants vs Array).
// TODO etudiant :
// 1. Pour chaque mode : construire le solver, mesurer t0 = Instant::now(), s.check(), t1
// 2. Repeter 100 fois, moyenner
// 3. Afficher : temps moyen Constants vs Array, verdict
println!("Exercice 3 a completer : mesurer et comparer les temps moyens Constants vs Array (100 runs).");

Exercice 3 a completer : mesurer et comparer les temps moyens Constants vs Array (100 runs).


## Conclusion

Ce notebook démontre le principe fondamental de la modélisation SMT : **le problème (les contraintes) est séparé de l'encodage (comment on les exprime)**. Deux encodages radicalement différents — 16 `Int` constantes vs un `Array(Int, Int)` — du même Sudoku 4x4 convergent vers la même solution valide.

| Aspect | Mode `Constants` | Mode `Array` |
|--------|------------------|--------------|
| Structure | 16 scalaires nommés | 1 fonction Index → Valeur |
| Accès cellule | `constantes()[r][c]` | `cellule(&grille, r, c)` (`select`) |
| Lisibilité grille fixe | Excellente | Moyenne |
| Cellule arbitraire / mise à jour | Malaisé (nom fixe) | Naturel (`Store`) |
| Sémantique (solutions) | **Identique** | **Identique** |

**Leçon** : choisir l'encodage selon le **contexte de raisonnement** (grille fixe vs dynamique), pas selon une prétendue « meilleure » représentation. Le solveur réécrit tout vers le même noyau SMT — la correction est garantie par les contraintes, pas par la syntaxe.

**Portage Rust — ce que la syntaxe change, et ce qu'elle ne change pas.** Le passage de pyz3 à la crate `z3` impose trois adaptations mécaniques (nommage `Symbol`, contrainte d'égalité `Ast::eq` au lieu de `==`, `select` qui renvoie un `Dynamic` à convertir par `as_int()`) ; il ne change **rien** au modèle, aux contraintes ni aux verdicts. C'est la démonstration, par le code, de la leçon ci-dessus.